# Vídeo para YouTube desde Colab

Abrir: https://colab.research.google.com/github/DreiSup/spanishnav.al-podcast/blob/claude/quirky-dirac-x18c3n/video/video.ipynb

Hace el `.mp4` de un episodio sin pasar por tu ordenador: **portada fija + master**, y lo deja en Drive, en la carpeta
`video/` del episodio (`docs/sop-publicacion.md`, apartado 5 bis).

**Entorno: CPU** (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → CPU*). No hace falta GPU.

**Uso:** pon el número del episodio en la celda 1 (`EPISODIO = "004"`) y *Entorno de ejecución → Ejecutar todas*. Drive pedirá permiso la primera vez.

Qué hace, y qué comprueba en cada paso (si algo falla, se para y lo dice):

1. Copia el master de Drive al disco de Colab y comprueba que se decodifica sin errores.
2. Hace el vídeo: 1920×1080, H.264, 25 fps; audio AAC-LC estéreo a 48 kHz y 192 kbps (lo que recomienda YouTube).
3. Comprueba el resultado: duraciones, que vídeo y audio se decodifican sin errores y que **el audio del vídeo es
   el del master** (lo compara en 12 tramos repartidos por todo el episodio).
4. Te deja escuchar un trozo **aquí mismo**. No juzgues el sonido por la vista previa de Drive: recién subido, Drive
   lo reproduce a calidad baja mientras lo procesa.
5. Lo copia a Drive y espera a que termine de sincronizarse.

Tarda aproximadamente un cuarto de la duración del episodio (7 min de audio, ~2 min).

In [ ]:
# 1. Qué episodio: su número, tal cual ("004"). Para un extra sin número, su carpeta ("2019/extra-on-meditation").
EPISODIO = "004"
SOBRESCRIBIR = False   # True = rehace el vídeo aunque ya exista en Drive

In [ ]:
# 2. Drive, ffmpeg y la portada
import subprocess, shutil, time
from pathlib import Path
import numpy as np
from google.colab import drive
drive.mount("/content/drive")

if not shutil.which("ffmpeg"):
    subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=True)
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])

TRABAJO = Path("/content/video_trabajo")
shutil.rmtree(TRABAJO, ignore_errors=True)
TRABAJO.mkdir()
PORTADA = TRABAJO / "portada-youtube.jpg"
subprocess.run(["curl", "-sSfL", "-o", str(PORTADA), "https://spanishnav.al/portada-youtube.jpg"], check=True)

def sonda(ruta, campos, flujo=None):
    cmd = ["ffprobe", "-v", "error"] + (["-select_streams", flujo] if flujo else []) + \
          ["-show_entries", campos, "-of", "default=nw=1", str(ruta)]
    salida = subprocess.run(cmd, capture_output=True, text=True, check=True).stdout
    return dict(l.split("=", 1) for l in salida.strip().splitlines() if "=" in l)

def errores_al_decodificar(ruta, mapa):
    r = subprocess.run(["ffmpeg", "-v", "error", "-i", str(ruta), "-map", mapa, "-f", "null", "-"],
                       capture_output=True, text=True)
    return (r.stderr.strip() or ("ffmpeg terminó con código %d" % r.returncode if r.returncode else ""))

p = sonda(PORTADA, "stream=codec_name,width,height", "v:0")
print("Portada:", p, f"{PORTADA.stat().st_size} bytes")

In [ ]:
# 3. El master: buscarlo en Drive, copiarlo a Colab y comprobarlo
RAIZ = Path("/content/drive/MyDrive/naval-podcast/transcripts")
EPISODIO = str(EPISODIO).strip()
if EPISODIO.isdigit():
    numero = f"{int(EPISODIO):03d}"
    candidatas = sorted(p for p in RAIZ.glob(f"20*/{numero} *") if p.is_dir())
    assert len(candidatas) == 1, f"Esperaba una carpeta '{numero} …' en {RAIZ}/<año>/, hay {len(candidatas)}: {candidatas}"
    CARPETA = candidatas[0]
    masters = sorted((CARPETA / "audio").glob(f"{numero}-*.mp3"))
else:
    CARPETA = RAIZ / EPISODIO
    masters = sorted((CARPETA / "audio").glob(f"{CARPETA.name}.mp3"))
assert len(masters) == 1, f"Esperaba un master en {CARPETA / 'audio'}, hay {len(masters)}: {[m.name for m in masters]}"
MASTER_DRIVE = masters[0]
anio, nombre = CARPETA.parent.name, MASTER_DRIVE.stem
DESTINO = CARPETA / "video" / f"{nombre}.mp4"
assert SOBRESCRIBIR or not DESTINO.exists(), f"El vídeo ya existe: {DESTINO}. Pon SOBRESCRIBIR = True para rehacerlo."

# Se trabaja con una copia local: leer de Drive mientras ffmpeg codifica es más lento y menos fiable.
MASTER = TRABAJO / MASTER_DRIVE.name
shutil.copyfile(MASTER_DRIVE, MASTER)
assert MASTER.stat().st_size == MASTER_DRIVE.stat().st_size, "La copia del master no tiene el tamaño del original"

m = sonda(MASTER, "stream=codec_name,sample_rate,channels,bit_rate:format=duration", "a:0")
DUR = float(m["duration"])
fallo = errores_al_decodificar(MASTER, "0:a:0")
assert not fallo, f"El master de Drive tiene errores al decodificarlo:\n{fallo[:2000]}"
print("Carpeta:", CARPETA.name)
print(f"Master:  {MASTER.name} · {m['codec_name']} · {m['sample_rate']} Hz · {m['channels']} canal(es) · "
      f"{int(m.get('bit_rate', 0) or 0) // 1000} kbps · {DUR / 60:.1f} min · sin errores")

In [ ]:
# 4. Hacer el vídeo
SALIDA = TRABAJO / f"{nombre}.mp4"
VF = "scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2,format=yuv420p"
t0 = time.time()
r = subprocess.run([
    "ffmpeg", "-v", "error", "-y",
    "-loop", "1", "-framerate", "25", "-i", str(PORTADA),
    "-i", str(MASTER),
    "-map", "0:v:0", "-map", "1:a:0", "-t", f"{DUR:.3f}",
    "-vf", VF, "-r", "25", "-c:v", "libx264", "-preset", "veryfast", "-tune", "stillimage", "-crf", "20",
    "-c:a", "aac", "-b:a", "192k", "-ar", "48000", "-ac", "2",
    "-movflags", "+faststart", str(SALIDA)], capture_output=True, text=True)
assert r.returncode == 0 and SALIDA.is_file(), f"ffmpeg falló:\n{r.stderr[-3000:]}"
if r.stderr.strip():
    print("Avisos de ffmpeg:\n" + r.stderr.strip()[-2000:])
print(f"Hecho en {time.time() - t0:.0f} s · {SALIDA.stat().st_size / 1e6:.1f} MB")

In [ ]:
# 5. Comprobar el vídeo
v = sonda(SALIDA, "stream=codec_name,width,height,pix_fmt,avg_frame_rate,duration", "v:0")
a = sonda(SALIDA, "stream=codec_name,profile,sample_rate,channels,duration", "a:0")
print("Vídeo:", v)
print("Audio:", a)
assert (v["codec_name"], v["width"], v["height"], v["pix_fmt"]) == ("h264", "1920", "1080", "yuv420p"), "Formato de vídeo inesperado"
assert (a["codec_name"], a["sample_rate"], a["channels"]) == ("aac", "48000", "2"), "Formato de audio inesperado"
assert abs(float(v["duration"]) - DUR) <= 0.5 and abs(float(a["duration"]) - DUR) <= 0.5, \
    f"Duraciones: master {DUR:.2f} s, vídeo {v['duration']}, audio {a['duration']}"
for mapa, que in (("0:v:0", "vídeo"), ("0:a:0", "audio")):
    fallo = errores_al_decodificar(SALIDA, mapa)
    assert not fallo, f"El {que} del .mp4 tiene errores al decodificarlo:\n{fallo[:2000]}"

# ¿El audio del vídeo es el del master? 12 tramos de 10 s repartidos por todo el episodio: se alinean y se mide
# cuánto se parecen (SNR, en dB). Una codificación AAC buena da 20–35 dB; un corte o un salto dan cerca de 0 desde ese punto,
# y un audio distorsionado, menos de 10. Probado el 6-nov con un vídeo estropeado a propósito.
def tramo(ruta, desde, mapa):
    crudo = subprocess.run(["ffmpeg", "-v", "error", "-ss", f"{desde:.3f}", "-t", "10", "-i", str(ruta),
                            "-map", mapa, "-ac", "1", "-ar", "48000", "-f", "f32le", "-"],
                           capture_output=True, check=True).stdout
    return np.frombuffer(crudo, dtype=np.float32)

snr = []
for desde in np.linspace(1, max(1, DUR - 12), 12):
    x, y = tramo(MASTER, desde, "0:a:0"), tramo(SALIDA, desde, "0:a:0")
    n = min(len(x), len(y))
    x, y = x[:n].astype(np.float64), y[:n].astype(np.float64)
    if n < 48000 or np.sqrt(np.mean(x ** 2)) < 1e-3:
        continue                                   # tramo demasiado corto o en silencio
    corr = np.fft.irfft(np.fft.rfft(x, 2 * n) * np.conj(np.fft.rfft(y, 2 * n)))
    lag = int(np.argmax(np.concatenate([corr[-2400:], corr[:2401]]))) - 2400   # desfase de ±50 ms como mucho
    xs, ys = (x[lag:], y[:n - lag]) if lag >= 0 else (x[:n + lag], y[-lag:])
    g = np.dot(xs, ys) / np.dot(ys, ys)
    snr.append(10 * np.log10(np.sum(xs ** 2) / np.sum((xs - g * ys) ** 2)))
print("SNR por tramo (dB):", " ".join(f"{s:.0f}" for s in snr))
assert len(snr) >= 6, "Demasiados tramos en silencio para comparar"
assert np.median(snr) >= 20 and min(snr) >= 12, \
    f"El audio del vídeo no se parece al master (mediana {np.median(snr):.1f} dB, mínimo {min(snr):.1f} dB): no lo subas"
print("OK: el vídeo cumple el formato y su audio es el del master")

In [ ]:
# 6. Escuchar un trozo aquí (30 s a partir del minuto 1, sacados del .mp4 ya hecho)
from IPython.display import Audio, display
MUESTRA = TRABAJO / "muestra.wav"
subprocess.run(["ffmpeg", "-v", "error", "-y", "-ss", str(min(60, max(0, DUR - 30))), "-t", "30", "-i", str(SALIDA),
                "-map", "0:a:0", str(MUESTRA)], check=True)
display(Audio(str(MUESTRA)))

In [ ]:
# 7. Copiar a Drive (carpeta video/ del episodio) y esperar a que se sincronice
DESTINO.parent.mkdir(exist_ok=True)
temporal = DESTINO.with_suffix(".mp4.parcial")
shutil.copyfile(SALIDA, temporal)
temporal.replace(DESTINO)
assert DESTINO.stat().st_size == SALIDA.stat().st_size, "La copia en Drive no tiene el tamaño del vídeo"
drive.flush_and_unmount()   # no cierres el cuaderno antes de que termine esta celda
print("En Drive:", f"naval-podcast/transcripts/{anio}/{CARPETA.name}/video/{DESTINO.name}")
print("Siguiente: pide a la sesión de dirección el título y la descripción, y sube el vídeo en YouTube Studio.")